In [20]:
import glob
import re
import ipywidgets as widgets
from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np
import yt

yt.set_log_level(50)

# 1. Gather and sort plot directories
plot_dirs = sorted(glob.glob("../raw/test*"))


def extract_step(path):
    match = re.search(r"test(\d+)", path)
    return int(match.group(1)) if match else -1


plot_dirs.sort(key=extract_step)

if not plot_dirs:
    raise FileNotFoundError("No directories matching '../raw/test*' were found.")

# Load datasets
ds_list = [yt.load(d) for d in plot_dirs]
times = [float(ds.current_time) for ds in ds_list]

# Get z-bounds from the first dataset
z_min = float(ds_list[0].domain_left_edge[2])
z_max = float(ds_list[0].domain_right_edge[2])

# Map field tuples to dropdown options
raw_fields = ds_list[0].field_list
field_options = [(f[1] if isinstance(f, tuple) else str(f), f) for f in raw_fields]

# 2. Build widgets
step_slider = widgets.IntSlider(
    value=0,
    min=0,
    max=len(plot_dirs) - 1,
    step=1,
    description="Step Index:",
    continuous_update=False,
)

z_slider = widgets.FloatSlider(
    value=(z_min + z_max) / 2.0,
    min=z_min,
    max=z_max,
    step=(z_max - z_min) / 100.0,
    description="z-Slice (m):",
    continuous_update=False,
)

field_dropdown = widgets.Dropdown(
    options=field_options,
    description="Field:",
)


# 3. Render function
def plot_slice(step_idx, z_val, field):
    ds = ds_list[step_idx]
    current_t = times[step_idx]

    p = yt.SlicePlot(ds, "z", field, center=[0.0, 0.0, z_val])

    # Force linear scale so yt doesn't attempt symlog linthresh estimation on zero/flat data
    p.set_log(field, False)

    # Compute symmetric limits around 0.0 for the current slice
    slc = ds.slice("z", z_val)
    data = slc[field].v
    finite_data = data[np.isfinite(data)]

    if len(finite_data) > 0 and not np.all(finite_data == 0):
        max_abs = float(np.max(np.abs(finite_data)))
    else:
        max_abs = 1.0  # Fallback range for uniform zero slice

    # Apply colormap, symmetric limits around 0, and reduce figure dimensions
    p.set_cmap(field, "berlin")
    p.set_zlim(field, -max_abs, max_abs)
    p.set_figure_size(5.0)

    p.annotate_title(f"t = {current_t:.4f} | z = {z_val:.4f}")
    p.show()


# 4. Interactive Display
interactive_plot = widgets.interactive(
    plot_slice, step_idx=step_slider, z_val=z_slider, field=field_dropdown
)

display(interactive_plot)

interactive(children=(IntSlider(value=0, continuous_update=False, description='Step Index:', max=1000), FloatS…